# 06 - Survivorship and Golden Record
# Enterprise BFSI Client MDM Platform

##Purpose:
   1. Read current Entity Resolution results
   2. Read current Stewardship decisions
   3. Build accepted MDM relationships
   4. Resolve connected components / entity clusters
   5. Assign stable MASTER_CLIENT_ID values
   6. Apply attribute-level survivorship
   7. Write Golden Record
   8. Write CLIENT_SOURCE_MAP
   9. Write survivorship audit information

##Initialize

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "bfsi_mdm_dev"

GOLDEN_ATTRIBUTES = [
    "name",
    "dob",
    "phone",
    "email",
    "address",
    "tax_id",
    "lei",
    "segment",
    "kyc_status",
    "verification_date",
    "aum",
    "risk_profile",
    "relationship_manager",
]

MATCH_DECISION = "MATCH"
APPROVED_DECISION = "APPROVE_MATCH"

print("Notebook 06 initialized.")
print(f"Catalog: {CATALOG}")
print(f"Golden attributes: {len(GOLDEN_ATTRIBUTES)}")

Notebook 06 initialized.
Catalog: bfsi_mdm_dev
Golden attributes: 13


##Verify the input tables

In [0]:
# Verify the objects that Notebook 06 depends on.

print("MDM tables:")
spark.sql(
    f"SHOW TABLES IN {CATALOG}.mdm"
).show(truncate=False)

print("Silver tables:")
spark.sql(
    f"SHOW TABLES IN {CATALOG}.silver"
).show(truncate=False)

MDM tables:
+--------+---------------------+-----------+
|database|tableName            |isTemporary|
+--------+---------------------+-----------+
|mdm     |candidate_pairs      |false      |
|mdm     |match_results        |false      |
|mdm     |review_queue         |false      |
|mdm     |stewardship_cases    |false      |
|mdm     |stewardship_decisions|false      |
+--------+---------------------+-----------+

Silver tables:
+--------+-----------------------+-----------+
|database|tableName              |isTemporary|
+--------+-----------------------+-----------+
|silver  |all_clients            |false      |
|silver  |core                   |false      |
|silver  |crm                    |false      |
|silver  |kyc                    |false      |
|silver  |stg_core_standardized  |false      |
|silver  |stg_crm_standardized   |false      |
|silver  |stg_kyc_standardized   |false      |
|silver  |stg_wealth_standardized|false      |
|silver  |wealth                 |false      |
+--

##Verify the current Silver population

In [0]:
silver = (
    spark.table(
        f"{CATALOG}.silver.all_clients"
    )
    .withColumn(
        "record_key",
        F.concat_ws(
            "::",
            F.col("source_system"),
            F.col("source_id")
        )
    )
)

silver_count = silver.count()

print(
    f"Current Silver records: {silver_count:,}"
)

Current Silver records: 2,638,869


##Verify source-record uniqueness

In [0]:
duplicate_record_keys = (
    silver
    .groupBy("record_key")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = (
    duplicate_record_keys
    .limit(1)
    .count()
)

assert duplicate_count == 0, (
    "Duplicate source record keys detected in Silver."
)

print(
    "Source record key uniqueness: PASS"
)

Source record key uniqueness: PASS


## Load Entity Resolution results

In [0]:
match_results = spark.table(
    f"{CATALOG}.mdm.match_results"
)

print(
    f"Match result rows: "
    f"{match_results.count():,}"
)

Match result rows: 138,074,372


##Load stewardship decisions

In [0]:
stewardship_decisions = spark.table(
    f"{CATALOG}.mdm.stewardship_decisions"
)

silver_count = silver.count()

# Each physical source record must have one unique key.
duplicate_keys = (
    silver
    .groupBy("record_key")
    .count()
    .filter(F.col("count") > 1)
)

assert duplicate_keys.limit(1).count() == 0, (
    "Duplicate source record keys detected in Silver."
)


print(
    f"Stewardship decision rows: "
    f"{stewardship_decisions.count():,}"
)

display(
    stewardship_decisions.select(
        "case_id",
        "record_key_a",
        "record_key_b",
        "decision",
        "decision_reason"
    )
)
print(f"Stewardship decisions: {stewardship_decisions.count():,}")
print("Input validation: PASS")

Stewardship decision rows: 2


case_id,record_key_a,record_key_b,decision,decision_reason
ab23452e74b8f641feb7082a1a91bfe1e1ccacd531a3099204c285aab53ad969,CORE::C00240191,CRM::CRM00164741,APPROVE_MATCH,Same client based on strong supporting identity evidence.
91bc2e588369057d3abeb308a99217cfd826bef818935f7792df449337545377,CRM::CRM00618369,KYC::KYC00760578,REJECT_MATCH,Conflicting identity evidence indicates different clients.


Stewardship decisions: 2
Input validation: PASS


##Build accepted MDM edges

MATCH
    ↓
accepted automatically

APPROVE_MATCH
    ↓
accepted after steward review

REJECT_MATCH
    ↓
not accepted

REVIEW
    ↓
not accepted

Represent every pair in a deterministic orientation so A-B and B-A are treated as the same edge.
Steward-approved REVIEW relationships

In [0]:
# ------------------------------------------------------------
# Helper:
# Canonicalize each pair so A-B and B-A are the same edge.
# ------------------------------------------------------------

def canonicalize_edges(df):
    return (
        df
        .select(
            F.least(
                F.col("record_key_a"),
                F.col("record_key_b")
            ).alias("record_key_a"),

            F.greatest(
                F.col("record_key_a"),
                F.col("record_key_b")
            ).alias("record_key_b")
        )
        .filter(
            F.col("record_key_a").isNotNull()
            &
            F.col("record_key_b").isNotNull()
            &
            (
                F.col("record_key_a")
                != F.col("record_key_b")
            )
        )
        .dropDuplicates()
    )


# ------------------------------------------------------------
# Automatic MATCH relationships
# ------------------------------------------------------------

automatic_edges = canonicalize_edges(
    match_results.filter(
        F.col("decision") == MATCH_DECISION
    )
)


# ------------------------------------------------------------
# Steward-approved REVIEW relationships
# ------------------------------------------------------------
# Notebook 05's stewardship_decisions table already contains
# the persisted stewardship decisions.
#
# APPROVE_MATCH -> accepted
# REJECT_MATCH  -> excluded
#
# No status filter is needed here.

approved_edges = canonicalize_edges(
    stewardship_decisions.filter(
        F.col("decision") == APPROVED_DECISION
    )
)


# ------------------------------------------------------------
# Final accepted identity graph
# ------------------------------------------------------------

accepted_edges = (
    automatic_edges
    .unionByName(approved_edges)
    .dropDuplicates()
)

print(
    f"Automatic MATCH edges : "
    f"{automatic_edges.count():,}"
)

print(
    f"Approved edges        : "
    f"{approved_edges.count():,}"
)

print(
    f"Accepted unique edges: "
    f"{accepted_edges.count():,}"
)

Automatic MATCH edges : 2,249,077
Approved edges        : 1
Accepted unique edges: 2,249,078


##Validate edge endpoints

In [0]:
nodes = silver.select("record_key").distinct()

missing_a = (
    accepted_edges
    .join(
        nodes,
        accepted_edges.record_key_a == nodes.record_key,
        "left_anti"
    )
)

missing_b = (
    accepted_edges
    .join(
        nodes,
        accepted_edges.record_key_b == nodes.record_key,
        "left_anti"
    )
)

assert missing_a.limit(1).count() == 0, (
    "Accepted edge contains missing record_key_a."
)

assert missing_b.limit(1).count() == 0, (
    "Accepted edge contains missing record_key_b."
)

print("Accepted-edge endpoint validation: PASS")

Accepted-edge endpoint validation: PASS


###Connected-components function

Conceptually

This:

A-B

B-C

becomes:

A

B

C

↓

same component

That is the bridge between pairwise matching and enterprise-level mastered entities.

In [0]:
def compute_connected_components(
    nodes_df,
    edges_df,
    max_iterations=50
):
    """
    Compute connected components using native Spark.

    Every node starts with itself as the component label.

    Example:

        A-B
        B-C

    Initial:
        A -> A
        B -> B
        C -> C

    After propagation:
        A -> A
        B -> A
        C -> A

    Therefore all three belong to one component.
    """

    # --------------------------------------------------------
    # Convert edge list into an undirected graph.
    # A-B becomes:
    #   A -> B
    #   B -> A
    # --------------------------------------------------------

    undirected_edges = (
        edges_df
        .select(
            F.col("record_key_a").alias("from_key"),
            F.col("record_key_b").alias("to_key")
        )
        .unionByName(
            edges_df.select(
                F.col("record_key_b").alias("from_key"),
                F.col("record_key_a").alias("to_key")
            )
        )
        .dropDuplicates()
    )

    # --------------------------------------------------------
    # Every node initially forms its own component.
    # --------------------------------------------------------

    labels = (
        nodes_df
        .select("record_key")
        .distinct()
        .withColumn(
            "component_key",
            F.col("record_key")
        )
    )

    # --------------------------------------------------------
    # Propagate the smallest component label.
    # --------------------------------------------------------

    for iteration in range(1, max_iterations + 1):

        proposals = (
            undirected_edges
            .join(
                labels,
                F.col("from_key") == F.col("record_key")
            )
            .select(
                F.col("to_key").alias("record_key"),
                F.col("component_key").alias(
                    "proposed_component"
                )
            )
        )

        new_labels = (
            labels
            .select(
                "record_key",
                F.col("component_key").alias(
                    "proposed_component"
                )
            )
            .unionByName(proposals)
            .groupBy("record_key")
            .agg(
                F.min("proposed_component")
                .alias("component_key")
            )
        )

        changed = (
            new_labels.alias("new")
            .join(
                labels.alias("old"),
                "record_key"
            )
            .filter(
                F.col("new.component_key")
                !=
                F.col("old.component_key")
            )
            .limit(1)
            .count()
        )

        print(
            f"Iteration {iteration}: "
            f"labels_changed={changed}"
        )

        labels = new_labels

        if changed == 0:
            print("Connected components converged.")
            return labels

    raise RuntimeError(
        f"Connected components did not converge "
        f"within {max_iterations} iterations."
    )

###Tiny graph test

connected records
    → one component

separate graph
    → separate component

isolated record
    → singleton component

In [0]:
# Small synthetic graph used ONLY to test the clustering algorithm.
#
# A-B and B-C should become one component.
# D-E should become another.
# F should remain a singleton.

test_nodes = spark.createDataFrame(
    [
        ("A",),
        ("B",),
        ("C",),
        ("D",),
        ("E",),
        ("F",),
    ],
    ["record_key"]
)

test_edges = spark.createDataFrame(
    [
        ("A", "B"),
        ("B", "C"),
        ("D", "E"),
    ],
    ["record_key_a", "record_key_b"]
)

test_labels = compute_connected_components(
    test_nodes,
    test_edges
)

display(
    test_labels.orderBy("record_key")
)

Iteration 1: labels_changed=1
Iteration 2: labels_changed=1
Iteration 3: labels_changed=0
Connected components converged.


record_key,component_key
A,A
B,A
C,A
D,D
E,D
F,F


###Real FULL clustering

Now we use the actual accepted MDM graph

In [0]:
cluster_labels = compute_connected_components(
    nodes,
    accepted_edges,
    max_iterations=50
)

cluster_members = (
    cluster_labels
    .select(
        "record_key",
        "component_key"
    )
)

silver_with_cluster = (
    silver
    .join(
        cluster_members,
        "record_key",
        "left"
    )
)

assert (
    silver_with_cluster.count()
    ==
    silver_count
), (
    "Silver → cluster join changed the record count."
)

assert (
    silver_with_cluster
    .filter(F.col("component_key").isNull())
    .limit(1)
    .count()
    == 0
), (
    "Some Silver records have no cluster."
)

print("Real clustering complete.")
print("Silver → cluster reconciliation: PASS")

Iteration 1: labels_changed=1
Iteration 2: labels_changed=1
Iteration 3: labels_changed=1
Iteration 4: labels_changed=1
Iteration 5: labels_changed=0
Connected components converged.
Real clustering complete.
Silver → cluster reconciliation: PASS


###Load previous master registry

In [0]:
previous_source_map_exists = spark.catalog.tableExists(
    f"{CATALOG}.gold.client_source_map"
)

if previous_source_map_exists:

    previous_source_map = (
        spark.table(
            f"{CATALOG}.gold.client_source_map"
        )
        .withColumn(
            "record_key",
            F.concat_ws(
                "::",
                F.col("source_system"),
                F.col("source_id")
            )
        )
        .select(
            "record_key",
            "master_client_id"
        )
    )

    duplicate_previous = (
        previous_source_map
        .groupBy("record_key")
        .count()
        .filter(F.col("count") > 1)
    )

    assert (
        duplicate_previous
        .limit(1)
        .count()
        == 0
    ), (
        "Previous CLIENT_SOURCE_MAP contains "
        "multiple masters for one source record."
    )

    print("Previous master registry found.")

else:

    previous_source_map = (
        spark.createDataFrame(
            [],
            "record_key string, master_client_id string"
        )
    )

    print(
        "No previous master registry found. "
        "Initial Gold run."
    )

No previous master registry found. Initial Gold run.


###Assign or reuse MASTER_CLIENT_ID

This is the main stability logic.

Case 1
New cluster
→ create new MC ID

Case 2
Cluster already known
→ reuse previous MC ID

Case 3
Cluster contains multiple historical MC IDs
→ retain one + flag merge warning

In [0]:
# ------------------------------------------------------------
# Find previously known masters within each current cluster.
# ------------------------------------------------------------

cluster_registry = (
    silver_with_cluster
    .select(
        "component_key",
        "record_key"
    )
    .join(
        previous_source_map,
        "record_key",
        "left"
    )
    .groupBy("component_key")
    .agg(
        F.countDistinct(
            "master_client_id"
        ).alias("known_master_count"),

        F.min(
            "master_client_id"
        ).alias("existing_master_id")
    )
)

# ------------------------------------------------------------
# Find highest master ID already used.
# ------------------------------------------------------------

max_master_number = (
    previous_source_map
    .select(
        F.max(
            F.regexp_extract(
                F.col("master_client_id"),
                r"(\d+)$",
                1
            ).cast("long")
        ).alias("max_number")
    )
    .first()["max_number"]
)

if max_master_number is None:
    max_master_number = 0

# ------------------------------------------------------------
# Existing clusters:
# reuse their existing ID.
# ------------------------------------------------------------

existing_assignments = (
    cluster_registry
    .filter(
        F.col("known_master_count") > 0
    )
    .select(
        "component_key",

        F.col("existing_master_id")
        .alias("master_client_id"),

        (
            F.col("known_master_count") > 1
        ).alias(
            "master_merge_warning"
        )
    )
)

# ------------------------------------------------------------
# New clusters:
# assign IDs after the previous maximum.
# ------------------------------------------------------------

new_clusters = (
    cluster_registry
    .filter(
        F.col("known_master_count") == 0
    )
)

new_window = Window.orderBy(
    "component_key"
)

new_assignments = (
    new_clusters
    .withColumn(
        "new_sequence",
        F.row_number().over(new_window)
    )
    .withColumn(
        "master_client_id",
        F.format_string(
            "MC%06d",
            F.lit(max_master_number)
            +
            F.col("new_sequence")
        )
    )
    .withColumn(
        "master_merge_warning",
        F.lit(False)
    )
    .select(
        "component_key",
        "master_client_id",
        "master_merge_warning"
    )
)

cluster_assignments = (
    existing_assignments
    .unionByName(new_assignments)
)

print(
    f"Previous maximum master number: "
    f"{max_master_number}"
)
print("MASTER_CLIENT_ID assignment complete.")

Previous maximum master number: 0
MASTER_CLIENT_ID assignment complete.


/databricks/spark/python/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


###Validate master assignment

In [0]:
# ============================================================
# VALIDATE MASTER ASSIGNMENT
# ============================================================

expected_silver_count = spark.table(
    f"{CATALOG}.silver.all_clients"
).count()

validation = (
    silver_master
    .agg(
        F.count("*").alias("silver_count"),
        F.countDistinct("record_key").alias("distinct_record_keys"),
        F.countDistinct("master_client_id").alias("master_count"),
        F.sum(
            F.when(
                F.col("master_client_id").isNull(),
                1
            ).otherwise(0)
        ).alias("null_master_count")
    )
    .collect()[0]
)

silver_count = validation["silver_count"]
record_key_count = validation["distinct_record_keys"]
master_count = validation["master_count"]
null_master_count = validation["null_master_count"]

print(f"Expected Silver records     : {expected_silver_count:,}")
print(f"Master-mapped records       : {silver_count:,}")
print(f"Distinct source records     : {record_key_count:,}")
print(f"Distinct MASTER_CLIENT_IDs  : {master_count:,}")
print(f"Null MASTER_CLIENT_IDs      : {null_master_count:,}")

# ------------------------------------------------------------
# Validation 1: all Silver records preserved
# ------------------------------------------------------------

assert silver_count == expected_silver_count, (
    f"Silver count mismatch: "
    f"{silver_count:,} != {expected_silver_count:,}"
)

# ------------------------------------------------------------
# Validation 2: every source record is unique
# ------------------------------------------------------------

assert record_key_count == silver_count, (
    "Duplicate record_key values detected"
)

# ------------------------------------------------------------
# Validation 3: every source record has a master
# ------------------------------------------------------------

assert null_master_count == 0, (
    f"Found {null_master_count:,} records without MASTER_CLIENT_ID"
)

print()
print("Silver record preservation: PASS")
print("Source record uniqueness: PASS")
print("MASTER_CLIENT_ID completeness: PASS")

/databricks/spark/python/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Expected Silver records     : 2,638,869
Master-mapped records       : 2,638,869
Distinct source records     : 2,638,869
Distinct MASTER_CLIENT_IDs  : 1,080,592
Null MASTER_CLIENT_IDs      : 0

Silver record preservation: PASS
Source record uniqueness: PASS
MASTER_CLIENT_ID completeness: PASS


In [0]:
# ============================================================
# VALIDATE COMPONENT → MASTER MAPPING
# ============================================================

component_master_conflicts = (
    silver_master
    .groupBy("component_key")
    .agg(
        F.countDistinct("master_client_id").alias(
            "master_id_count"
        )
    )
    .filter(
        F.col("master_id_count") > 1
    )
)

conflict_count = component_master_conflicts.count()

print(
    f"Components mapped to multiple MASTER_CLIENT_IDs: "
    f"{conflict_count:,}"
)

assert conflict_count == 0, (
    f"Found {conflict_count:,} components with multiple "
    f"MASTER_CLIENT_ID values"
)

print("Component → MASTER_CLIENT_ID consistency: PASS")

/databricks/spark/python/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Components mapped to multiple MASTER_CLIENT_IDs: 0
Component → MASTER_CLIENT_ID consistency: PASS


/databricks/spark/python/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


###Build and validate CLIENT_SOURCE_MAP

In [0]:
client_source_map = (
    silver_master
    .select(
        "master_client_id",
        "source_system",
        "source_id"
    )
    .dropDuplicates()
)

duplicate_source_mappings = (
    client_source_map
    .groupBy(
        "source_system",
        "source_id"
    )
    .count()
    .filter(F.col("count") > 1)
)

assert (
    duplicate_source_mappings
    .limit(1)
    .count()
    == 0
), (
    "A source record maps to multiple masters."
)

assert (
    client_source_map.count()
    ==
    silver_count
), (
    "CLIENT_SOURCE_MAP does not cover all Silver records."
)

print("CLIENT_SOURCE_MAP validation: PASS")

/databricks/spark/python/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/spark/python/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


CLIENT_SOURCE_MAP validation: PASS


/databricks/spark/python/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


###Build all survivorship signals

This is where we combine several old cells into one meaningful transformation.

For every source record:

source_authority \
verification_flag \
recency_date \
completeness_score \
match_confidence 

The project policy is:

1. Source authority
2. Verification
3. Recency
4. Completeness
5. Match confidence
6. Stable source system
7. Stable source ID

The KYC > Core > CRM > Wealth ordering is our policy (not a universal banking rule).

In [0]:
# ============================================================
# CELL 12 — BUILD SURVIVORSHIP SIGNALS
# ============================================================

# ------------------------------------------------------------
# 1. Rebuild the mastered Silver view
# ------------------------------------------------------------

silver_for_survivorship = (
    spark.table(
        f"{CATALOG}.silver.all_clients"
    )
    .withColumn(
        "record_key",
        F.concat_ws(
            "::",
            F.col("source_system"),
            F.col("source_id")
        )
    )
)

silver_master = (
    silver_for_survivorship
    .join(
        cluster_members,
        "record_key",
        "left"
    )
    .join(
        cluster_assignments,
        "component_key",
        "left"
    )
)

# ------------------------------------------------------------
# 2. SOURCE AUTHORITY
# ------------------------------------------------------------

authority_map = F.create_map(
    F.lit("KYC"), F.lit(4),
    F.lit("CORE"), F.lit(3),
    F.lit("CRM"), F.lit(2),
    F.lit("WEALTH"), F.lit(1)
)

silver_master = (
    silver_master
    .withColumn(
        "source_authority",
        F.coalesce(
            authority_map[
                F.col("source_system")
            ],
            F.lit(0)
        )
    )

    # --------------------------------------------------------
    # Verification flag
    # Empty / null verification dates = not verified
    # --------------------------------------------------------

    .withColumn(
        "verification_flag",
        F.when(
            F.trim(
                F.coalesce(
                    F.col(
                        "normalized_verification_date"
                    ),
                    F.lit("")
                )
            ) != "",
            F.lit(1)
        ).otherwise(
            F.lit(0)
        )
    )

    # --------------------------------------------------------
    # Recency date
    #
    # IMPORTANT:
    # try_to_date prevents ANSI parsing failure on:
    # null, empty string, or malformed date.
    # --------------------------------------------------------

    .withColumn(
        "recency_date",
        F.coalesce(
            F.expr(
                "try_to_date(normalized_verification_date, 'yyyy-MM-dd')"
            ),
            F.lit(
                "1900-01-01"
            ).cast("date")
        )
    )
)

# ------------------------------------------------------------
# 3. COMPLETENESS SCORE
# ------------------------------------------------------------

presence_expressions = [
    F.when(
        F.col(attribute).isNotNull()
        &
        (
            F.trim(
                F.col(attribute).cast("string")
            ) != ""
        ),
        F.lit(1)
    ).otherwise(
        F.lit(0)
    )
    for attribute in GOLDEN_ATTRIBUTES
]

silver_master = (
    silver_master
    .withColumn(
        "completeness_score",
        sum(
            presence_expressions,
            F.lit(0)
        )
    )
)

# ------------------------------------------------------------
# 4. ACCEPTED MATCH CONFIDENCE
#
# MATCH         → accepted
# APPROVE_MATCH → accepted
# REJECT_MATCH  → excluded
# REVIEW        → excluded
# ------------------------------------------------------------

current_match_results = spark.table(
    f"{CATALOG}.mdm.match_results"
)

current_stewardship_decisions = spark.table(
    f"{CATALOG}.mdm.stewardship_decisions"
)

automatic_scores = (
    current_match_results
    .filter(
        F.col("decision") == MATCH_DECISION
    )
    .select(
        "record_key_a",
        "record_key_b",
        "match_score"
    )
)

approved_scores = (
    current_stewardship_decisions
    .filter(
        F.col("decision") == APPROVED_DECISION
    )
    .select(
        "record_key_a",
        "record_key_b",
        "match_score"
    )
)

accepted_scores = (
    automatic_scores
    .unionByName(
        approved_scores
    )
    .dropDuplicates(
        [
            "record_key_a",
            "record_key_b",
            "match_score"
        ]
    )
)

confidence_by_record = (
    accepted_scores
    .select(
        F.col("record_key_a").alias(
            "record_key"
        ),
        "match_score"
    )
    .unionByName(
        accepted_scores.select(
            F.col("record_key_b").alias(
                "record_key"
            ),
            "match_score"
        )
    )
    .groupBy(
        "record_key"
    )
    .agg(
        F.max(
            "match_score"
        ).alias(
            "match_confidence"
        )
    )
)

silver_master = (
    silver_master
    .join(
        confidence_by_record,
        "record_key",
        "left"
    )
    .withColumn(
        "match_confidence",
        F.coalesce(
            F.col("match_confidence"),
            F.lit(0.0)
        )
    )
)

print("Survivorship signals prepared.")

Survivorship signals prepared.


/databricks/spark/python/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


###Unpivot, rank and select attribute winners

Instead of maintaining separate code for every attribute, we turn the 13 wide columns into a long structure.

Why unpivot?

Before:

master
name
phone
email
address
...

After:

master | attribute | candidate value

That makes the ranking generic.

One rule handles:

name
phone
email
address
risk_profile

instead of writing separate ranking logic for each one.

In [0]:
attribute_structs = []

for attribute in GOLDEN_ATTRIBUTES:

    attribute_structs.append(
        F.struct(
            F.lit(attribute).alias(
                "attribute_name"
            ),
            F.col(attribute)
            .cast("string")
            .alias(
                "attribute_value"
            )
        )
    )

attribute_long = (
    silver_master
    .select(
        "master_client_id",
        "source_system",
        "source_id",
        "source_authority",
        "verification_flag",
        "recency_date",
        "completeness_score",
        "match_confidence",

        F.explode(
            F.array(*attribute_structs)
        ).alias("attribute")
    )
    .select(
        "master_client_id",
        "source_system",
        "source_id",
        "source_authority",
        "verification_flag",
        "recency_date",
        "completeness_score",
        "match_confidence",

        F.col(
            "attribute.attribute_name"
        ).alias("attribute_name"),

        F.col(
            "attribute.attribute_value"
        ).alias("attribute_value")
    )
)

# A blank value cannot win an attribute.
attribute_candidates = (
    attribute_long
    .filter(
        F.col("attribute_value").isNotNull()
        &
        (
            F.trim(
                F.col("attribute_value")
            ) != ""
        )
    )
)

survivorship_window = (
    Window
    .partitionBy(
        "master_client_id",
        "attribute_name"
    )
    .orderBy(
        F.desc("source_authority"),
        F.desc("verification_flag"),
        F.desc("recency_date"),
        F.desc("completeness_score"),
        F.desc("match_confidence"),
        F.asc("source_system"),
        F.asc("source_id")
    )
)

ranked_candidates = (
    attribute_candidates
    .withColumn(
        "survivorship_rank",
        F.row_number().over(
            survivorship_window
        )
    )
)

winners = (
    ranked_candidates
    .filter(
        F.col("survivorship_rank") == 1
    )
)

print("Attribute-level survivorship ranking complete.")

Attribute-level survivorship ranking complete.


/databricks/spark/python/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


###Build audit and Golden Record

Complete master × attribute universe \
This guarantees 13 rows per master even when an attribute has no populated candidate.\
Convert winning attributes back to one Golden Record row.\
The master × 13 universe matters
Suppose:

MC000123

has no LEI anywhere.

We still retain:

MC000123 | lei | NULL

with:

survivorship_reason = no populated candidate

That is much better than pretending the attribute never existed.


In [0]:
# ------------------------------------------------------------
# Complete master × attribute universe
#
# This guarantees 13 rows per master even when an attribute
# has no populated candidate.
# ------------------------------------------------------------

master_ids = (
    silver_master
    .select("master_client_id")
    .distinct()
)

attribute_dimension = spark.createDataFrame(
    [(a,) for a in GOLDEN_ATTRIBUTES],
    ["attribute_name"]
)

master_attribute_universe = (
    master_ids
    .crossJoin(attribute_dimension)
)

# ------------------------------------------------------------
# Survivorship audit
# ------------------------------------------------------------

survivorship_audit = (
    master_attribute_universe
    .join(
        winners,
        [
            "master_client_id",
            "attribute_name"
        ],
        "left"
    )
    .withColumn(
        "survivorship_reason",
        F.when(
            F.col("source_id").isNull(),
            F.lit("no populated candidate")
        )
        .otherwise(
            F.lit(
                "highest-ranked populated candidate "
                "using source authority, verification, "
                "recency, completeness, match confidence, "
                "and stable source identifiers"
            )
        )
    )
)

merge_warning_by_master = (
    cluster_assignments
    .groupBy("master_client_id")
    .agg(
        F.max(
            F.col(
                "master_merge_warning"
            ).cast("int")
        ).alias(
            "master_merge_warning"
        )
    )
)

survivorship_audit = (
    survivorship_audit
    .join(
        merge_warning_by_master,
        "master_client_id",
        "left"
    )
    .withColumn(
        "master_merge_warning",
        F.coalesce(
            F.col("master_merge_warning"),
            F.lit(0)
        )
    )
    .withColumn(
        "audit_timestamp",
        F.current_timestamp()
    )
)

# ------------------------------------------------------------
# Convert winning attributes back to one Golden Record row.
# ------------------------------------------------------------

golden_wide = (
    survivorship_audit
    .groupBy("master_client_id")
    .pivot(
        "attribute_name",
        GOLDEN_ATTRIBUTES
    )
    .agg(
        F.first("attribute_value")
    )
)

golden_wide = (
    golden_wide
    .withColumn(
        "aum",
        F.col("aum").cast("decimal(18,2)")
    )
)

master_client = golden_wide.select(
    "master_client_id",
    *GOLDEN_ATTRIBUTES
)

print("Survivorship audit and Golden Record built.")

Survivorship audit and Golden Record built.


/databricks/spark/python/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


###Write the three Gold tables

In [0]:
# ============================================================
# CELL 15 — WRITE GOLD TABLES
# Optimized write order:
#   1. CLIENT_SOURCE_MAP
#   2. SURVIVORSHIP_AUDIT
#   3. MASTER_CLIENT from persisted audit
# ============================================================

# ------------------------------------------------------------
# 1. Write CLIENT_SOURCE_MAP
# ------------------------------------------------------------

client_source_map.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"{CATALOG}.gold.client_source_map"
    )

print("CLIENT_SOURCE_MAP written.")

# ------------------------------------------------------------
# 2. Write SURVIVORSHIP_AUDIT
# ------------------------------------------------------------
# This materializes the expensive survivorship result once.

survivorship_audit.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"{CATALOG}.gold.survivorship_audit"
    )

print("SURVIVORSHIP_AUDIT written.")

# ------------------------------------------------------------
# 3. Rebuild MASTER_CLIENT from persisted audit table
# ------------------------------------------------------------
# This avoids recomputing the complete Silver → survivorship
# lineage for the Golden Record write.

persisted_audit = spark.table(
    f"{CATALOG}.gold.survivorship_audit"
)

master_client = (
    persisted_audit
    .groupBy("master_client_id")
    .pivot(
        "attribute_name",
        GOLDEN_ATTRIBUTES
    )
    .agg(
        F.first("attribute_value")
    )
    .withColumn(
        "aum",
        F.col("aum").cast("decimal(18,2)")
    )
    .select(
        "master_client_id",
        *GOLDEN_ATTRIBUTES
    )
)

# ------------------------------------------------------------
# 4. Write MASTER_CLIENT
# ------------------------------------------------------------

master_client.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"{CATALOG}.gold.master_client"
    )

print("MASTER_CLIENT written.")

print()
print("=" * 70)
print("NOTEBOOK 06 GOLD TABLES WRITTEN")
print("=" * 70)
print(
    f"CLIENT_SOURCE_MAP rows : "
    f"{spark.table(f'{CATALOG}.gold.client_source_map').count():,}"
)
print(
    f"SURVIVORSHIP_AUDIT rows: "
    f"{spark.table(f'{CATALOG}.gold.survivorship_audit').count():,}"
)
print(
    f"MASTER_CLIENT rows      : "
    f"{spark.table(f'{CATALOG}.gold.master_client').count():,}"
)

/databricks/spark/python/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


CLIENT_SOURCE_MAP written.


/databricks/spark/python/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


SURVIVORSHIP_AUDIT written.
MASTER_CLIENT written.

NOTEBOOK 06 GOLD TABLES WRITTEN
CLIENT_SOURCE_MAP rows : 2,638,869
SURVIVORSHIP_AUDIT rows: 14,047,696
MASTER_CLIENT rows      : 1,080,592
